# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos Semanas 7 y 8 : LDA y LLM audio-a-texto**

* **Nombres y matrículas:**

  *   Antonio Ramón Valerio Tejada - A01797448
  *   Claudia Suarez Segura - A01797641
  *   Rigoberto Bracamontes Salazar - A01134473

* **Número de Equipo:** 6


* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

In [45]:
!pip install -q openai-whisper
!apt-get -qq install -y ffmpeg
!pip install --upgrade transformers
!pip install -q -U transformers accelerate

In [46]:
import requests
import os
from urllib.parse import urlparse
import json

import librosa
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import re
import nltk
import string
import whisper

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM



# Descargar stopwords en español para limpieza de texto
nltk.download('stopwords')
from nltk.corpus import stopwords

# Stopwords en español
stop_words = set(stopwords.words('spanish'))

# Stopwords adicionales para eliminar palabras poco informativas
stop_words_extra = {
    "ser", "tener", "sino", "vez", "poder", "solo", "todas", "todos",
    "hacer", "mismo", "misma", "mayor", "mejor", "pues",
    "dijo", "repuso", "vio", "así", "cada", "gran"
}

stop_words = stop_words.union(stop_words_extra)

# La transcripción se realizará con Whisper
client = None

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# Verificamos si hay GPU disponible:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")


Usando dispositivo: cuda


# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [49]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...


def descargar_mp3(url, carpeta_destino="archivos"):


  try:
    # Crear carpeta
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)

    # Obtener nombre del archivo
    parsed_url = urlparse(url)
    nombre_archivo = os.path.basename(parsed_url.path)

    # Si no tiene extensión, agregar .mp3
    if not nombre_archivo.endswith('.mp3'):
        nombre_archivo += '.mp3'

    ruta_completa = os.path.join(carpeta_destino, nombre_archivo)

    # Descargar el archivo
    print(f"Descargando: {nombre_archivo}")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(ruta_completa, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"✓ Descargado: {ruta_completa}")
    return True

  except Exception as e:
    print(f"Error descargando {url}: {str(e)}")
    return False

audio_solicitados = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
urls_mp3 = [f"https://www.gutenberg.org/files/21144/mp3/21144-{x:02d}.mp3" for x in audio_solicitados]

for url in urls_mp3:
    descargar_mp3(url)

Descargando: 21144-01.mp3
✓ Descargado: archivos/21144-01.mp3
Descargando: 21144-04.mp3
✓ Descargado: archivos/21144-04.mp3
Descargando: 21144-05.mp3
✓ Descargado: archivos/21144-05.mp3
Descargando: 21144-06.mp3
✓ Descargado: archivos/21144-06.mp3
Descargando: 21144-14.mp3
✓ Descargado: archivos/21144-14.mp3
Descargando: 21144-22.mp3
✓ Descargado: archivos/21144-22.mp3
Descargando: 21144-24.mp3
✓ Descargado: archivos/21144-24.mp3
Descargando: 21144-25.mp3
✓ Descargado: archivos/21144-25.mp3
Descargando: 21144-26.mp3
✓ Descargado: archivos/21144-26.mp3
Descargando: 21144-27.mp3
✓ Descargado: archivos/21144-27.mp3


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

Para la extracción de texto de los audios se seleccionó **Whisper local** (`openai-whisper`) ejecutado directamente en Google Colab. La elección de este modelo se debió a que ofrece muy buen desempeño en tareas de transcripción automática, tiene soporte para idioma español, permite procesar archivos de audio de forma local y evita el uso de servicios de pago por API.

Inicialmente se exploraron alternativas basadas en modelos de reconocimiento de voz; sin embargo, Whisper local resultó ser la opción más conveniente para esta actividad por su facilidad de integración, precisión en la transcripción y viabilidad de ejecución dentro del entorno de Colab.

Además, este enfoque permitió trabajar directamente con los 10 audios solicitados y almacenar los resultados en una estructura tipo diccionario con formato clave-valor, por ejemplo:

`{audio_01: "texto transcrito...", audio_04: "texto transcrito...", ...}`

De esta forma, las transcripciones quedaron listas para las etapas posteriores de limpieza, extracción de palabras clave con LDA y generación de resúmenes y subtemas.

In [50]:
transcrip = {}

# Ruta donde están los audios descargados
folder_path = "/content/archivos"

# Verificar si se usará GPU o CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Cargar modelo Whisper local

modelo_whisper = "small" if device == "cuda" else "base"

print(f"Cargando modelo Whisper local: {modelo_whisper}")
model = whisper.load_model(modelo_whisper, device=device)

# Obtener archivos de la carpeta
files_in_folder = os.listdir(folder_path)

# Procesar cada archivo mp3
for file_name in sorted(files_in_folder):
    file_path = os.path.join(folder_path, file_name)

    if os.path.isfile(file_path) and file_name.lower().endswith(".mp3"):
        print(f"Procesando archivo: {file_path}")

        result = model.transcribe(
            file_path,
            language="es",
            fp16=(device == "cuda")
        )

        transcrip[file_name] = result["text"].strip()

print("\nFinished processing all files in the folder.")

# Cambiar nombres de llaves:
# de 21144-01.mp3 a audio_01, etc.
new_transcrip = {}

for key, value in transcrip.items():
    parts = key.split(".")[0].split("-")

    if len(parts) > 1:
        number_part = parts[-1]
        new_key = f"audio_{number_part[-2:]}"
        new_transcrip[new_key] = value
    else:
        new_transcrip[key] = value

transcrip = new_transcrip

# Mostrar transcripciones
for key, value in transcrip.items():
    print("\n" + "="*80)
    print(key)
    print(value)


Usando dispositivo: cuda
Cargando modelo Whisper local: small
Procesando archivo: /content/archivos/21144-01.mp3
Procesando archivo: /content/archivos/21144-04.mp3
Procesando archivo: /content/archivos/21144-05.mp3
Procesando archivo: /content/archivos/21144-06.mp3
Procesando archivo: /content/archivos/21144-14.mp3
Procesando archivo: /content/archivos/21144-22.mp3
Procesando archivo: /content/archivos/21144-24.mp3
Procesando archivo: /content/archivos/21144-25.mp3
Procesando archivo: /content/archivos/21144-26.mp3
Procesando archivo: /content/archivos/21144-27.mp3

Finished processing all files in the folder.

audio_01
Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula n. 61, el lobo y el cordero en el templo. Tándose cuenta de que era perseguido por un lobo, un pequeño cordelito decidió refugiarse en un templo cercano. Lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro, lo emolaría a su Dios. Mejor así, replicó el cordero. P

In [51]:
# Guardar transcripciones en JSON (Drive con fallback local)

drive_path = "/content/drive/MyDrive/NLP (1)/act 7 y 8/transcriptions_whisper_local.json"
local_path = "/content/transcriptions_whisper_local.json"

# Intentar guardar en Drive; si falla, guardar localmente
try:
    output_filename = drive_path
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(transcrip, f, ensure_ascii=False, indent=4)
    print(f"Transcripciones guardadas en Drive: {output_filename}")
except Exception as e:
    print(f"No se pudo guardar en Drive ({e}). Guardando localmente...")
    output_filename = local_path
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(transcrip, f, ensure_ascii=False, indent=4)
    print(f"Transcripciones guardadas localmente: {output_filename}")


Transcripciones guardadas en Drive: /content/drive/MyDrive/NLP (1)/act 7 y 8/transcriptions_whisper_local.json


In [52]:
# Cargar transcripciones desde JSON (Drive con fallback local)

drive_path = "/content/drive/MyDrive/NLP (1)/act 7 y 8/transcriptions_whisper_local.json"
local_path = "/content/transcriptions_whisper_local.json"

try:
    with open(drive_path, 'r', encoding='utf-8') as f:
        transcrip = json.load(f)
    print(f"Transcripciones cargadas desde Drive: {drive_path}")
except Exception:
    with open(local_path, 'r', encoding='utf-8') as f:
        transcrip = json.load(f)
    print(f"Transcripciones cargadas localmente: {local_path}")

print(f"Total de fábulas cargadas: {len(transcrip)}")
transcrip


Transcripciones cargadas desde Drive: /content/drive/MyDrive/NLP (1)/act 7 y 8/transcriptions_whisper_local.json
Total de fábulas cargadas: 10


{'audio_01': 'Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula n. 61, el lobo y el cordero en el templo. Tándose cuenta de que era perseguido por un lobo, un pequeño cordelito decidió refugiarse en un templo cercano. Lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro, lo emolaría a su Dios. Mejor así, replicó el cordero. Prefiero ser víctima para un Dios, a tener que perecer en tus colmillos. Si sin remedio vamos a ser sacrificados, más nos vale que sea con el mayor honor. Fin de la fábula. Esta es una grabación del dominio público.',
 'audio_04': 'Las fábulas de SOPO grabado para LibriVox.org por Roberto Antonio Muñoz, fábula número 64, el lobo y la grúda. A un lobo que comía un hueso, se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio. Encontró en su correr a una grulla y le pidió que le salvara de aquella situación y que enseguida le pagaría por ello. Aptó la grulla e introdujo su cabeza

# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [53]:
def limpiar_transcripcion(texto):
    # Normalizar espacios
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Eliminar encabezado común hasta después de "fábula número ##"
    texto = re.sub(
        r'^.*?f[aá]bula\s+(?:n\.?|n[uú]mero)?\s*\d+\s*[\.,:;-]?\s*',
        '',
        texto,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Eliminar final común desde "fin de la fábula"
    texto = re.sub(
        r'\s*fin\s+de(?:\s+la)?\s+f[aá]bula\.?.*$',
        '',
        texto,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Por si queda la frase de dominio público sin "fin de la fábula"
    texto = re.sub(
        r'\s*esta\s+es\s+una\s+grabaci[oó]n\s+del\s+dominio\s+p[uú]blico.*$',
        '',
        texto,
        flags=re.IGNORECASE | re.DOTALL
    )

    return texto.strip(" .,\n")


# Aplicar limpieza a cada transcripción
transcripciones_limpias = {
    k: limpiar_transcripcion(v)
    for k, v in transcrip.items()
}

# Evidencia: mostrar inicio y final antes/después para las 3 primeras fábulas
print("=" * 70)
print("VERIFICACIÓN DE LIMPIEZA (inicio y final de cada fábula)")
print("=" * 70)
for clave in list(transcripciones_limpias.keys())[:3]:
    original = transcrip[clave]
    limpio   = transcripciones_limpias[clave]
    print(f"\n>>> {clave}")
    print(f"  [ORIGINAL  inicio]: {original[:120]!r}")
    print(f"  [ORIGINAL  final ]: {original[-80:]!r}")
    print(f"  [LIMPIO    inicio]: {limpio[:120]!r}")
    print(f"  [LIMPIO    final ]: {limpio[-80:]!r}")

print("\n" + "=" * 70)
print("TEXTO LIMPIO COMPLETO")
print("=" * 70)
for clave, texto in transcripciones_limpias.items():
    print(f"\n{clave}:\n{texto}\n{chr(45)*60}")


VERIFICACIÓN DE LIMPIEZA (inicio y final de cada fábula)

>>> audio_01
  [ORIGINAL  inicio]: 'Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula n. 61, el lobo y el cordero en el t'
  [ORIGINAL  final ]: 'con el mayor honor. Fin de la fábula. Esta es una grabación del dominio público.'
  [LIMPIO    inicio]: 'el lobo y el cordero en el templo. Tándose cuenta de que era perseguido por un lobo, un pequeño cordelito decidió refugi'
  [LIMPIO    final ]: 'Si sin remedio vamos a ser sacrificados, más nos vale que sea con el mayor honor'

>>> audio_04
  [ORIGINAL  inicio]: 'Las fábulas de SOPO grabado para LibriVox.org por Roberto Antonio Muñoz, fábula número 64, el lobo y la grúda. A un lobo'
  [ORIGINAL  final ]: 's si te dejan sano y salvo. Fin de fábula. Esta grabación es de dominio público.'
  [LIMPIO    inicio]: 'el lobo y la grúda. A un lobo que comía un hueso, se le atragantó el hueso en la garganta y corría por todas partes en b'
  [LIMPIO    final ]

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

Antes de aplicar el modelo LDA para la extracción de palabras clave, se realizó un proceso de limpieza lingüística sobre las transcripciones. Este paso es importante porque LDA es sensible al ruido textual y puede verse afectado por palabras muy frecuentes que no aportan valor temático.

En la función `limpiar_texto`, primero se convierten todos los caracteres a minúsculas para evitar duplicidad entre palabras con distinta capitalización. Después, se eliminan números y signos de puntuación, ya que no aportan información semántica relevante para el análisis. Posteriormente, el texto se divide en palabras y se eliminan las stopwords del español, como "el", "la", "de" y "y". También se descartan palabras con longitud menor o igual a dos caracteres, debido a que suelen aportar poca información temática.

Adicionalmente, se amplió el conjunto de stopwords con términos propios del encabezado de los audios de LibriVox/Gutenberg (por ejemplo: `sopo`, `librivoxorg`, `wwwpaulinoinfo`, nombres de locutores como `paulino`, `roberto`, `karen`). Estos términos pueden filtrarse hasta el corpus si la limpieza de inicio/final del Ejercicio 2b no los elimina completamente en alguna fábula. Esto garantiza que LDA trabaje únicamente con el contenido narrativo relevante.

Se decidió conservar las palabras restantes sin aplicar lematización o stemming, ya que cada fábula es relativamente corta y una limpieza excesiva podría eliminar información útil para la identificación de tópicos.


In [54]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

# Stopwords adicionales: palabras del encabezado de LibriVox/Gutenberg
# y nombres de locutores que pueden filtrarse desde las transcripciones
stop_words_extra = {
    "sopo", "esopo", "librivoxorg", "librivox", "grabado", "grabada",
    "fabula", "numero", "paulino", "roberto",
    "alejandro", "karen", "savage", "zavic", "ochito", "wwwpaulinoinfo",
    "dominio", "publico"
}
stop_words = stop_words.union(stop_words_extra)


def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'\d+', '', texto)          # eliminar números
    texto = re.sub(r'[' + string.punctuation + '\u00bf\u00a1]', '', texto)  # quitar puntuación
    palabras = texto.split()
    palabras_limpias = [p for p in palabras if p not in stop_words and len(p) > 2]
    return ' '.join(palabras_limpias)


# **Ejercicio 4:**

En este fragmento se aplica el modelo LDA (Latent Dirichlet Allocation) de forma individual a cada fábula para extraer sus palabras clave más representativas. Primero, el texto limpio se vectoriza mediante un modelo de bolsa de palabras, y posteriormente se configura LDA con un único tópico por fábula, de acuerdo con las instrucciones de la actividad.

A partir del tópico identificado, se extraen las 20 palabras con mayor peso dentro del modelo. Este procedimiento se repite para cada una de las 10 fábulas solicitadas, almacenando los resultados en un diccionario para su posterior análisis y visualización.

In [55]:
# Ejercicio 4: Extracción de palabras clave con LDA por fábula

n_top_words = 20
palabras_clave = {}

for nombre, texto_original in transcripciones_limpias.items():
    texto = limpiar_texto(texto_original)

    # Vectorización (bolsa de palabras)
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform([texto])
    vocab_size = X.shape[1]

    # Si el vocabulario es menor al número deseado, ajustar dinámicamente
    # para no perder ninguna fábula del análisis
    k = min(n_top_words, vocab_size)
    if vocab_size < n_top_words:
        print(f"{nombre}: vocabulario reducido ({vocab_size} palabras únicas), "
              f"extrayendo top-{k} en lugar de {n_top_words}")

    # LDA con 1 solo tópico por fábula
    lda = LatentDirichletAllocation(n_components=1, max_iter=10, random_state=42)
    lda.fit(X)

    # Palabras clave del tema principal
    palabras = vectorizer.get_feature_names_out()
    tema = lda.components_[0]
    indices_top = tema.argsort()[-k:][::-1]
    top_words = [palabras[i] for i in indices_top]
    palabras_clave[nombre] = top_words

    print(f"\n{nombre} ({vocab_size} palabras únicas):")
    print("Palabras clave:", ', '.join(top_words))

print(f"\nTotal de fábulas procesadas: {len(palabras_clave)}/10")



audio_01 (28 palabras únicas):
Palabras clave: lobo, templo, dios, cordero, tándose, sacrificados, sacrificador, replicó, remedio, víctima, vale, vamos, perseguido, prefiero, refugiarse, pequeño, llamó, honor, encontraba, perecer

audio_04 (48 palabras únicas):
Palabras clave: lobo, paga, hueso, pidió, grulla, garganta, cabeza, boca, salva, salvara, traficantes, suficiente, sana, salvo, situación, sano, nunca, oye, pagaría, partes

audio_05 (38 palabras únicas):
Palabras clave: caballo, cebada, lobo, ruido, siguió, sembrado, pasaba, preferido, rato, parezca, masticarla, malvado, oídos, oír, hallado, gusto, estómago, encontró, dientes, dejó

audio_06 (46 palabras únicas):
Palabras clave: lobo, ley, asno, repártelo, épocas, rey, primero, partes, orejas, ordenando, moviendo, repartiese, pusieran, propias, lobos, magnífica, manera, llegas, leyes, legislar

audio_14 (26 palabras únicas):
Palabras clave: lobo, cabrito, serenamente, seguridad, replicó, proveen, protegido, poderosos, valor, s

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

Este fragmento utiliza un modelo de lenguaje grande (LLM) para generar, a partir de las palabras clave obtenidas con LDA, un resumen y tres posibles subtemas para cada fábula. Para esta etapa se seleccionó el modelo **Qwen2.5-Instruct**, ejecutado localmente en Google Colab, con el objetivo de evitar el uso de APIs de pago y mantener el procesamiento dentro del entorno de trabajo.

Para cada fábula se construyó un prompt que incluye las palabras clave extraídas mediante LDA y el texto limpio de la fábula. El texto se incluye como contexto de apoyo para reducir el riesgo de generar información no presente en la transcripción. El modelo devuelve un único enunciado que resume la fábula y tres subtemas relacionados con su contenido.

Finalmente, las respuestas generadas se almacenan en un diccionario para facilitar su consulta posterior. Además, se realizó una revisión final de los resultados para asegurar que cada fábula cumpliera con el formato solicitado: un resumen de una sola oración y tres subtemas breves y coherentes.


In [56]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

# Selección del modelo según disponibilidad de GPU
modelo_llm_local = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")
print(f"Cargando modelo: {modelo_llm_local}")

tokenizer = AutoTokenizer.from_pretrained(modelo_llm_local)

model_llm = AutoModelForCausalLM.from_pretrained(
    modelo_llm_local,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

if device == "cpu":
    model_llm = model_llm.to(device)


def generar_texto_local(prompt, max_new_tokens=220):
    messages = [
        {
            "role": "system",
            "content": (
                "Eres un asistente experto en análisis de fábulas de Esopo. "
                "Respondes siempre en español, de forma breve, clara y estructurada. "
                "No inventas personajes ni acciones que no estén en el texto."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1200
    ).to(device)

    with torch.no_grad():
        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    respuesta = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return respuesta.strip()


# Usar directamente el diccionario de transcripciones limpias definido en Ejercicio 2b
textos_base = transcripciones_limpias
print(f"Textos base cargados: {len(textos_base)} fábulas")


def generar_resumen_y_subtemas(palabras_clave, textos_base):
    resultados = {}

    for nombre, lista in palabras_clave.items():
        keywords = ", ".join(lista)
        texto_fabula = textos_base.get(nombre, "")

        prompt = f"""
Analiza la siguiente fábula de Esopo en español.

Texto de la fábula:
{texto_fabula}

Palabras clave extraídas mediante LDA:
{keywords}

Con base principalmente en las palabras clave LDA y usando el texto de la fábula solo para evitar inventar detalles, genera exactamente:
1. Un único enunciado que describa o resuma la fábula.
2. Tres posibles subtemas relacionados con la fábula.

Reglas:
- Responde solo en español.
- No inventes personajes, acciones ni lugares que no aparezcan en el texto.
- No expliques el proceso.
- El resumen debe ser una sola oración.
- Cada subtema debe ser una frase corta.
- Usa exactamente el siguiente formato:

Resumen: ...
Subtema 1: ...
Subtema 2: ...
Subtema 3: ...
"""

        salida = generar_texto_local(prompt, max_new_tokens=220)

        resultados[nombre] = {
            "palabras_clave": lista,
            "resumen_y_subtemas": salida
        }

    return resultados


resultados = generar_resumen_y_subtemas(palabras_clave, textos_base)

# Mostrar resultados crudos del LLM (antes de revisión manual)
print("\nResultados generados por el LLM (pendientes de revisión):")
for nombre, contenido in resultados.items():
    print("\n" + "="*80)
    print(nombre)
    print("Palabras clave:", ", ".join(contenido["palabras_clave"]))
    print(contenido["resumen_y_subtemas"])


Usando dispositivo: cuda
Cargando modelo: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Textos base cargados: 10 fábulas

Resultados generados por el LLM (pendientes de revisión):

audio_01
Palabras clave: lobo, templo, dios, cordero, tándose, sacrificados, sacrificador, replicó, remedio, víctima, vale, vamos, perseguido, prefiero, refugiarse, pequeño, llamó, honor, encontraba, perecer
Resumen: El cordero se refugia en un templo mientras un lobo lo persigue. El cordero dice que prefieren ser sacrificados al dios del lobo sobre morir en sus colmillos. Finalmente, ambos son sacrificados pero el cordero es considerado como víctima honorable debido a su decisión.

Subtema 1: La persecución del cordero por parte del lobo.
Subtema 2: El cordero decide refugiarse en el templo.
Subtema 3: El sacrificador del cordero y el lobo.

audio_04
Palabras clave: lobo, paga, hueso, pidió, grulla, garganta, cabeza, boca, salva, salvara, traficantes, suficiente, sana, salvo, situación, sano, nunca, oye, pagaría, partes
Resumen: El lobo atragantó un hueso mientras comía y buscaba ayuda. Una gr

A partir de la salida generada por el LLM local, se realizó una **revisión manual** para asegurar que:
- El resumen de cada fábula sea exactamente **un único enunciado** claro y coherente.
- Los tres subtemas sean frases breves y consistentes con el texto original.
- No se incluyan personajes, acciones o lugares que no estén presentes en la transcripción.

La celda siguiente contiene los resultados finales ya validados, que reemplazan la salida automática del LLM donde fue necesario ajustar el formato o corregir información.


In [57]:
resultados_revisados = {
    "audio_01": {
        "palabras_clave": palabras_clave["audio_01"],
        "resumen_y_subtemas": """Resumen: Un cordero perseguido por un lobo prefiere refugiarse en un templo y aceptar el sacrificio antes que morir devorado.\nSubtema 1: El peligro ante un depredador.\nSubtema 2: La elección entre dos destinos adversos.\nSubtema 3: El refugio como forma de protección."""
    },
    "audio_04": {
        "palabras_clave": palabras_clave["audio_04"],
        "resumen_y_subtemas": """Resumen: Un lobo atragantado con un hueso pide ayuda a una grulla, pero luego se niega a pagarle por haberle salvado, considerando que no haberla devorado es suficiente recompensa.\nSubtema 1: La ingratitud del lobo.\nSubtema 2: El riesgo de ayudar a los malvados.\nSubtema 3: La falsa superioridad."""
    },
    "audio_05": {
        "palabras_clave": palabras_clave["audio_05"],
        "resumen_y_subtemas": """Resumen: Un lobo intenta engañar a un caballo hablándole de cebada, pero el caballo desconfía de sus verdaderas intenciones y no cae en la trampa.\nSubtema 1: El engaño disfrazado de amabilidad.\nSubtema 2: La prudencia ante los malvados.\nSubtema 3: La diferencia entre palabras y acciones."""
    },
    "audio_06": {
        "palabras_clave": palabras_clave["audio_06"],
        "resumen_y_subtemas": """Resumen: Un lobo que se proclama rey impone una ley de reparto equitativo, pero un asno lo expone como hipócrita, revelando que la ley se basa en el poder del más fuerte.\nSubtema 1: La ley impuesta por la fuerza.\nSubtema 2: El abuso de poder.\nSubtema 3: La desigualdad en el reparto."""
    },
    "audio_14": {
        "palabras_clave": palabras_clave["audio_14"],
        "resumen_y_subtemas": """Resumen: Un cabrito protegido en un lugar seguro insulta al lobo, demostrando que su valor depende de la protección que tiene.\nSubtema 1: La seguridad como fuente de valentía.\nSubtema 2: La falsa superioridad ante el peligro.\nSubtema 3: El valor condicionado por el contexto."""
    },
    "audio_22": {
        "palabras_clave": palabras_clave["audio_22"],
        "resumen_y_subtemas": """Resumen: Un perro confunde una almeja con un huevo y la traga sin reflexionar, sufriendo las consecuencias de su error.\nSubtema 1: La impulsividad al tomar decisiones.\nSubtema 2: Las apariencias engañosas.\nSubtema 3: La importancia de reflexionar antes de actuar."""
    },
    "audio_24": {
        "palabras_clave": palabras_clave["audio_24"],
        "resumen_y_subtemas": """Resumen: Un perro pierde su propio trozo de carne al intentar arrebatar el supuesto pedazo ajeno que ve reflejado en el río.\nSubtema 1: La codicia y sus consecuencias.\nSubtema 2: El engaño de las apariencias.\nSubtema 3: La pérdida de lo propio por desear lo ajeno."""
    },
    "audio_25": {
        "palabras_clave": palabras_clave["audio_25"],
        "resumen_y_subtemas": """Resumen: Un perro roba un trozo de carne mientras el carnicero está distraído y logra escapar antes de ser detenido.\nSubtema 1: La distracción del carnicero.\nSubtema 2: La astucia del perro.\nSubtema 3: La vigilancia después de una pérdida."""
    },
    "audio_26": {
        "palabras_clave": palabras_clave["audio_26"],
        "resumen_y_subtemas": """Resumen: Un perro con campanilla presume públicamente sin entender que la señal no representa virtud sino advertencia por su mala conducta.\nSubtema 1: La falsa presunción.\nSubtema 2: La diferencia entre fama y virtud.\nSubtema 3: Las señales públicas de mala conducta."""
    },
    "audio_27": {
        "palabras_clave": palabras_clave["audio_27"],
        "resumen_y_subtemas": """Resumen: Un perro que persigue a un león retrocede al escucharlo rugir, mostrando que no siempre se debe enfrentar aquello que no se puede soportar.\nSubtema 1: La imprudencia ante un peligro mayor.\nSubtema 2: El miedo frente a la fuerza real.\nSubtema 3: La importancia de conocer los propios límites."""
    }
}

for nombre, contenido in resultados_revisados.items():
    print("\n" + "="*80)
    print(nombre)
    print("Palabras clave:", ", ".join(contenido["palabras_clave"]))
    print(contenido["resumen_y_subtemas"])


audio_01
Palabras clave: lobo, templo, dios, cordero, tándose, sacrificados, sacrificador, replicó, remedio, víctima, vale, vamos, perseguido, prefiero, refugiarse, pequeño, llamó, honor, encontraba, perecer
Resumen: Un cordero perseguido por un lobo prefiere refugiarse en un templo y aceptar el sacrificio antes que morir devorado.
Subtema 1: El peligro ante un depredador.
Subtema 2: La elección entre dos destinos adversos.
Subtema 3: El refugio como forma de protección.

audio_04
Palabras clave: lobo, paga, hueso, pidió, grulla, garganta, cabeza, boca, salva, salvara, traficantes, suficiente, sana, salvo, situación, sano, nunca, oye, pagaría, partes
Resumen: Un lobo atragantado con un hueso pide ayuda a una grulla, pero luego se niega a pagarle por haberle salvado, considerando que no haberla devorado es suficiente recompensa.
Subtema 1: La ingratitud del lobo.
Subtema 2: El riesgo de ayudar a los malvados.
Subtema 3: La falsa superioridad.

audio_05
Palabras clave: caballo, cebada, 

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**



Los resultados de esta actividad permitieron integrar diferentes etapas de procesamiento de lenguaje natural a partir de archivos de audio: descarga de datos, transcripción automática, limpieza textual, extracción de palabras clave mediante LDA y generación de resúmenes con un modelo LLM.

Para la etapa audio-a-texto se utilizó **Whisper local** ejecutado en Google Colab, lo cual permitió transcribir los audios en español sin depender de APIs de pago. En general, el modelo logró obtener transcripciones suficientemente claras para continuar con el análisis, aunque se identificaron algunos retos relacionados con acentos, posibles errores de segmentación y pequeñas diferencias en la forma en que se reconocen ciertas palabras.

En la etapa de limpieza se eliminaron elementos que no aportaban información temática, como signos de puntuación, números, palabras vacías y términos demasiado cortos. Esta limpieza fue importante para mejorar la calidad de las palabras clave obtenidas con LDA. Sin embargo, debido a que cada fábula es un texto breve, se evitó aplicar una limpieza excesiva para no perder información útil.

El uso de LDA permitió identificar palabras representativas de cada fábula, considerando un tópico por documento. Posteriormente, estas palabras clave se utilizaron junto con el texto limpio como entrada para un modelo LLM local (**Qwen2.5-Instruct**), con el objetivo de generar un resumen y tres subtemas por fábula. Esta combinación permitió sintetizar el contenido principal de cada historia y relacionarlo con posibles temas narrativos o morales.

Uno de los principales retos fue controlar la generación del LLM, ya que en algunos casos el modelo podía agregar información no presente en el texto original. Por ello, se realizó una revisión final de las respuestas generadas para asegurar que cada resumen fuera un único enunciado y que los subtemas fueran coherentes con la fábula correspondiente.

En conclusión, la actividad permitió observar cómo las técnicas tradicionales de NLP, como limpieza de texto y LDA, pueden complementarse con modelos de lenguaje para transformar información proveniente de audio en resúmenes y temas interpretables. Esta metodología puede ser útil en aplicaciones de análisis documental, recuperación de información, clasificación temática y generación automática de contenido a partir de fuentes orales.


# **Fin de la actividad LDA y LLM: audio-a-texto**